# Sentiment Analysis Gradio App
## Compare TextBlob vs VADER vs Flair

In [1]:
# Install required packages
!pip install gradio textblob vaderSentiment flair torch pandas -q
!python -m textblob.download_corpora -q
print("✅ All packages installed!")


[notice] A new release of pip available: 22.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Finished.
✅ All packages installed!


[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\jesse\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\jesse\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\jesse\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\jesse\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package conll2000 to
[nltk_data]     C:\Users\jesse\AppData\Roaming\nltk_data...
[nltk_data]   Package conll2000 is already up-to-date!
[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\jesse\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is alr

In [2]:
import gradio as gr
import pandas as pd
from textblob import TextBlob
from textblob.sentiments import NaiveBayesAnalyzer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from flair.models import TextClassifier
from flair.data import Sentence

print("✅ All libraries imported!")

c:\Users\jesse\OneDrive - LYCEE Jules Haag\Formation-Dev-IA\Mes projets\NLP\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All libraries imported!


In [3]:
# Load Flair model
print("Loading Flair model...")
classifier = TextClassifier.load('en-sentiment')
print("✅ Flair model loaded!")

Loading Flair model...
✅ Flair model loaded!


In [4]:
# Initialize VADER
vader_analyzer = SentimentIntensityAnalyzer()

# Define sentiment analysis functions

def textblob_pattern(text):
    """TextBlob Pattern Analyzer - Lexicon based"""
    blob = TextBlob(str(text))
    polarity = blob.sentiment.polarity
    # Convert to 0-1 scale
    score = (polarity + 1) / 2
    return score, "Positive" if polarity > 0 else "Negative" if polarity < 0 else "Neutral"

def textblob_naive_bayes(text):
    """TextBlob Naive Bayes Analyzer - Statistical"""
    try:
        blob = TextBlob(str(text), analyzer=NaiveBayesAnalyzer())
        classification = blob.sentiment.classification
        pos_prob = blob.sentiment.p_pos
        return pos_prob, "Positive" if classification == 'pos' else "Negative"
    except:
        return 0.5, "Neutral"

def vader_sentiment(text):
    """VADER Sentiment Analyzer - Lexicon + Rules"""
    scores = vader_analyzer.polarity_scores(str(text))
    compound = scores['compound']
    # Convert to 0-1 scale
    score = (compound + 1) / 2
    return score, "Positive" if compound > 0 else "Negative" if compound < 0 else "Neutral"

def flair_sentiment(text):
    """Flair Sentiment Analyzer - Deep Learning"""
    try:
        sentence = Sentence(str(text))
        classifier.predict(sentence)
        label = sentence.labels[0]
        score = label.score
        return score, label.value
    except:
        return 0.5, "NEUTRAL"

print("✅ All functions defined!")

✅ All functions defined!


In [5]:
def analyze_sentiment(text):
    """
    Analyze text with all 4 models and return results
    """
    if not text or len(text.strip()) == 0:
        return "Please enter some text!", "", ""
    
    # Get predictions from all models
    tb_pattern_score, tb_pattern_label = textblob_pattern(text)
    tb_nb_score, tb_nb_label = textblob_naive_bayes(text)
    vader_score, vader_label = vader_sentiment(text)
    flair_score, flair_label = flair_sentiment(text)
    
    # Create results dataframe
    results = pd.DataFrame({
        'Model': [
            '🔷 TextBlob (Pattern)',
            '🟢 TextBlob (Naive Bayes)',
            '🟡 VADER',
            '🔵 Flair (Deep Learning)'
        ],
        'Sentiment': [tb_pattern_label, tb_nb_label, vader_label, flair_label],
        'Confidence': [f"{tb_pattern_score:.2%}", f"{tb_nb_score:.2%}", 
                      f"{vader_score:.2%}", f"{flair_score:.2%}"]
    })
    
    # Find best model (highest confidence)
    scores = [tb_pattern_score, tb_nb_score, vader_score, flair_score]
    models = [
        'TextBlob (Pattern)',
        'TextBlob (Naive Bayes)',
        'VADER',
        'Flair (Deep Learning)'
    ]
    
    best_idx = scores.index(max(scores))
    best_model = models[best_idx]
    best_score = max(scores)
    
    summary = f"🏆 Best Model: {best_model}\n📊 Confidence: {best_score:.2%}"
    
    # Create detailed info
    details = f"""
    ### Model Comparison:
    
    **TextBlob (Pattern Analyzer):**
    - Sentiment: {tb_pattern_label}
    - Confidence: {tb_pattern_score:.2%}
    - Type: Lexicon-based (dictionary lookup)
    
    **TextBlob (Naive Bayes):**
    - Sentiment: {tb_nb_label}
    - Confidence: {tb_nb_score:.2%}
    - Type: Statistical (trained on movie reviews)
    
    **VADER:**
    - Sentiment: {vader_label}
    - Confidence: {vader_score:.2%}
    - Type: Lexicon + Rules (handles intensity, negations)
    
    **Flair (Deep Learning):**
    - Sentiment: {flair_label}
    - Confidence: {flair_score:.2%}
    - Type: Transformer-based neural network (best accuracy)
    """
    
    return results, summary, details

print("✅ Analysis function ready!")

✅ Analysis function ready!


In [6]:
# Create Gradio Interface

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🎯 Sentiment Analysis Comparison Tool
    
    **Compare 4 different sentiment analysis models on your text:**
    - TextBlob (Pattern) - Lexicon-based
    - TextBlob (Naive Bayes) - Statistical
    - VADER - Lexicon + Rules
    - Flair - Deep Learning 🏆
    
    Try analyzing movie reviews, product reviews, or any text!
    """)
    
    # Input section
    with gr.Row():
        text_input = gr.Textbox(
            label="📝 Enter your text:",
            placeholder="e.g., This movie is absolutely fantastic! Best film ever.",
            lines=3
        )
    
    # Button
    analyze_btn = gr.Button("🔍 Analyze Sentiment", variant="primary")
    
    # Output sections
    gr.Markdown("### 📊 Results:")
    
    with gr.Row():
        results_table = gr.Dataframe(
            label="Model Comparison",
            interactive=False
        )
    
    with gr.Row():
        best_model_output = gr.Markdown()
    
    with gr.Row():
        details_output = gr.Markdown()
    
    # Examples
    gr.Markdown("### 💡 Try these examples:")
    gr.Examples(
        examples=[
            "This is absolutely amazing! I love it!",
            "Terrible product. Complete waste of money.",
            "It's okay, nothing special.",
            "This movie was so bad I couldn't even finish it.",
            "Incredible service and outstanding quality!"
        ],
        inputs=text_input,
        outputs=[results_table, best_model_output, details_output],
        fn=analyze_sentiment,
        cache_examples=False
    )
    
    # Connect button to analysis function
    analyze_btn.click(
        fn=analyze_sentiment,
        inputs=text_input,
        outputs=[results_table, best_model_output, details_output]
    )

print("✅ Gradio interface created!")

C:\Users\jesse\AppData\Local\Temp\ipykernel_35224\2759510017.py:3: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


✅ Gradio interface created!


In [ ]:
# Launch the app
demo.launch(share=True, debug=True)

* Running on local URL:  http://127.0.0.1:7863

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/06/03 14:46:08 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: i/o timeout
